# LearnWise LLM server: Qwen3-8B on a free Kaggle T4

This notebook runs **Qwen3-8B** (4-bit quantized, ~5 GB) with **Ollama** on Kaggle's free GPU and makes it reachable
from your laptop through a free **Cloudflare tunnel**. The backend then talks to it like any OpenAI-compatible API.

**Before you run:**
1. Right panel → *Session options* → **Accelerator: GPU T4 x2** (a single T4 is enough; x2 is fine).
2. **Internet: On** (needs a phone-verified Kaggle account).
3. *Run All*. The last cells print a URL like `https://something.trycloudflare.com/v1`. Put that in `backend/.env` as `LLM_BASE_URL`.

**Keep in mind:** Kaggle gives ~30 GPU hours/week per account and sessions end after 12 h (or when idle), and the
URL **changes every time** you restart. Anyone with the URL can use your model, so don't post it publicly, and
stop the session when you're done to save GPU hours.

## 1. Settings

In [ ]:
MODEL = "qwen3:8b"         # alternatives: "qwen3:4b" (faster), "qwen3:14b" (better, slower, still fits a T4)
CONTEXT_LENGTH = 8192      # tokens the model can read at once; RAG prompts need room for retrieved passages
TUNNEL = "cloudflare"      # "cloudflare" (no account needed) or "ngrok" (needs NGROK_AUTHTOKEN in Kaggle Secrets)
KEEP_ALIVE_HOURS = 6       # how long the last cell keeps the session busy

## 2. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 3. Install and start Ollama

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time, requests

env = dict(os.environ,
           OLLAMA_HOST="0.0.0.0:11434",
           OLLAMA_KEEP_ALIVE="-1",                    # keep the model loaded in GPU memory
           OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
           OLLAMA_NUM_PARALLEL="2")                   # you and Sana can send requests at the same time
ollama = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/version", timeout=2); print("Ollama is running"); break
    except requests.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start, see /kaggle/working/ollama.log")

## 4. Download the model (~5 GB, takes 1-3 minutes)

In [ ]:
!ollama pull {MODEL}

## 5. Warm up and test locally

In [ ]:
t = time.time()
r = requests.post("http://localhost:11434/v1/chat/completions", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "In one sentence, what is retrieval-augmented generation? /no_think"}],
    "max_tokens": 100,
}, timeout=300)
print(r.json()["choices"][0]["message"]["content"])
print(f"took {time.time() - t:.1f}s (the first call also loads the model into GPU memory)")
!nvidia-smi --query-gpu=memory.used --format=csv

## 6. Open the public tunnel

In [ ]:
import re, threading

public_url = None

if TUNNEL == "cloudflare":
    subprocess.run("wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                   " && chmod +x /kaggle/working/cloudflared", shell=True, check=True)
    # --http-host-header: Ollama only accepts requests addressed to localhost, so the tunnel rewrites the Host header
    cf = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:11434",
                           "--http-host-header", "localhost:11434"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in cf.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            public_url = m.group(0); break
    # keep reading the log in the background so the pipe never fills up
    threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()

elif TUNNEL == "ngrok":
    subprocess.run("pip -q install pyngrok", shell=True, check=True)
    from kaggle_secrets import UserSecretsClient
    from pyngrok import ngrok
    ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_AUTHTOKEN"))
    public_url = ngrok.connect(11434, "http", host_header="localhost:11434").public_url

print("=" * 70)
print("Put these lines in backend/.env :")
print(f"LLM_BASE_URL={public_url}/v1")
print(f"LLM_MODEL={MODEL}")
print("=" * 70)

## 7. Test through the tunnel (same request your laptop will make)

In [ ]:
time.sleep(5)  # a new Cloudflare URL can take a few seconds to start working
r = requests.post(f"{public_url}/v1/chat/completions", json={
    "model": MODEL, "messages": [{"role": "user", "content": "Say hello in 5 words. /no_think"}], "max_tokens": 30,
}, timeout=120)
print(r.status_code, r.json()["choices"][0]["message"]["content"])

## 8. Keep the session alive

Leave this cell running while you work. Interrupt it (■) and stop the session when you are done.
If the tunnel dies, re-run section 6 to get a new URL.

In [ ]:
end = time.time() + KEEP_ALIVE_HOURS * 3600
while time.time() < end:
    try:
        ok = requests.get(f"{public_url}/v1/models", timeout=15).status_code == 200
    except Exception:
        ok = False
    print(time.strftime("%H:%M"), "tunnel OK" if ok else "tunnel NOT reachable", "|", public_url, flush=True)
    time.sleep(300)